# 01 Momentum Research — Macro Metals System

> **Strategy:** Canonical Time-Series Momentum (TSMOM)
> **Reference:** Moskowitz, Ooi & Pedersen (2012); Quantpedia; JPM/CME report
> **Scope:** In-sample development (2015–2022), monthly rebalance
> **Universe:** Multi-asset futures — commodities (70% risk budget), rates, equities, FX (30%)
> **Signal:** sign(12-month return), monthly rebalance, inverse-vol position sizing
> **Overlay:** Sector risk budgets + portfolio-level vol targeting (10% annual)
>
> Self-contained BQuant notebook — executable top-to-bottom.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import yaml
from pathlib import Path
from datetime import datetime
from math import sqrt

# Bloomberg BQL
import bql
bq = bql.Service()

print(f"Session started : {datetime.now():%Y-%m-%d %H:%M}")
print(f"BQL service     : {type(bq).__name__}")
print(f"NumPy {np.__version__}  |  pandas {pd.__version__}")

## Config & Parameters

Load `parameters.yaml` and `tickers.yaml`.

| Parameter | Value | Description |
|-----------|-------|-------------|
| `lookback_days` | 252 | 12-month return lookback |
| `rebalance_freq` | monthly | Signal + vol + weights frozen monthly |
| `target_vol_annual` | 0.10 | Per-instrument vol target |
| `vol_decay_lambda` | 0.94 | EWMA decay (RiskMetrics) |
| `tc_bp_per_side` | 2.0 | Transaction cost per side |
| `commodities_risk_budget` | 0.70 | Commodity sector risk share |
| `diversifiers_risk_budget` | 0.30 | Rates + equities + FX risk share |
| `portfolio_vol_target` | 0.10 | Portfolio-level vol overlay |
| `portfolio_scale_floor` | 0.05 | Min overlay scale (Fix 2) |
| `abs_weight_caps` | bucket-specific | Hard per-instrument caps (Fix 1) |

In [ ]:
CONFIG_DIR = Path("config")

with open(CONFIG_DIR / "parameters.yaml") as f:
    params = yaml.safe_load(f)
with open(CONFIG_DIR / "tickers.yaml") as f:
    tickers = yaml.safe_load(f)

gcfg    = params["global"]
targets = params["performance_targets"]

# ── Canonical TSMOM parameters ───────────────────────────────────
LOOKBACK_DAYS = 252
TARGET_VOL    = gcfg.get("target_portfolio_vol_annual", 0.10)
VOL_LAMBDA    = gcfg.get("vol_decay_lambda", 0.94)
LEV_CAP       = gcfg.get("vol_cap_multiplier", 2.0)
TC_BP         = 2.0

# Sector risk budgets (commodity-focused)
COMMODITY_RISK_BUDGET    = 0.70
DIVERSIFIER_RISK_BUDGET  = 0.30
PORTFOLIO_VOL_TARGET     = 0.10
PORTFOLIO_VOL_LOOKBACK   = 60

# ── FIX 2: Lower portfolio overlay scale floor ───────────────────
PORTFOLIO_SCALE_BOUNDS   = (0.05, 2.0)   # was (0.5, 2.0)

# ── FIX 1: Hard absolute weight caps per bucket ─────────────────
ABS_WEIGHT_CAP_BY_BUCKET = {
    "commodities": 1.0,
    "equities":    0.8,
    "fx":          0.8,
    "rates":       0.5,
}

# Optional per-instrument overrides (Fix 6: energy)
ABS_WEIGHT_CAP_BY_INSTRUMENT = {
    "cl_fut_front": 0.5,
    "ng_fut_front": 0.5,
}

# Winsorization threshold for return cleaning (Fix 5)
WINSOR_Z_THRESHOLD = 5.0

IS_START = gcfg["in_sample_start"]
IS_END   = gcfg["in_sample_end"]

print("Canonical TSMOM parameters:")
print(f"  Lookback           : {LOOKBACK_DAYS}d (12 months)")
print(f"  Rebalance          : monthly")
print(f"  Per-inst vol target: {TARGET_VOL:.0%}")
print(f"  EWMA lambda        : {VOL_LAMBDA}")
print(f"  Leverage cap (rel) : {LEV_CAP:.1f}x")
print(f"  TC per side        : {TC_BP:.1f} bp")
print(f"  Risk budget        : {COMMODITY_RISK_BUDGET:.0%} commodities / {DIVERSIFIER_RISK_BUDGET:.0%} diversifiers")
print(f"  Portfolio vol tgt  : {PORTFOLIO_VOL_TARGET:.0%}")
print(f"  Overlay scale floor: {PORTFOLIO_SCALE_BOUNDS[0]} (FIX 2)")
print(f"  Abs weight caps    : {ABS_WEIGHT_CAP_BY_BUCKET} (FIX 1)")
print(f"  Energy overrides   : {ABS_WEIGHT_CAP_BY_INSTRUMENT} (FIX 6)")
print(f"  Winsor z-threshold : {WINSOR_Z_THRESHOLD} (FIX 5)")
print(f"  IS period          : {IS_START} to {IS_END}")

## Universe Expansion

Multi-asset futures universe in 4 buckets, matching the canonical TSMOM literature.

| Bucket | Role | Risk Budget |
|--------|------|-------------|
| **Commodities** | Precious metals, energy, agriculture, base metals | 70% |
| **Rates** | SOFR strip + UST futures | 30% combined |
| **Equities** | Major equity index futures | |
| **FX** | G10 liquid pairs | |

Tickers resolved from `tickers.yaml`; missing instruments logged and skipped.

In [ ]:
# ── Multi-asset TSMOM universe ────────────────────────────────────
universe = {
    "commodities": [
        "gc_fut_front", "si_fut_front", "pl_fut_front",
        "hg_fut_front",
        "cl_fut_front", "ng_fut_front",
        "w_fut_front", "c_fut_front", "s_fut_front",
    ],
    "rates": [
        "sofr_fut_front", "sofr_fut_second", "sofr_fut_third", "sofr_fut_fourth",
        "ust_2y_fut", "ust_5y_fut", "ust_10y_fut", "ust_30y_fut",
    ],
    "equities": [
        "es_fut_front", "nq_fut_front", "stoxx50_fut_front", "nikkei_fut_front",
    ],
    "fx": [
        "eurusd_spot", "usdjpy_spot", "gbpusd_spot", "audusd_spot",
        "nzdusd_spot", "usdcad_spot",
    ],
}

LABELS = {
    "gc_fut_front": "Gold (GC)", "si_fut_front": "Silver (SI)",
    "pl_fut_front": "Platinum (PL)", "hg_fut_front": "Copper (HG)",
    "cl_fut_front": "Crude Oil (CL)", "ng_fut_front": "Nat Gas (NG)",
    "w_fut_front": "Wheat (W)", "c_fut_front": "Corn (C)", "s_fut_front": "Soybeans (S)",
    "sofr_fut_front": "SOFR 1st", "sofr_fut_second": "SOFR 2nd",
    "sofr_fut_third": "SOFR 3rd", "sofr_fut_fourth": "SOFR 4th",
    "ust_2y_fut": "UST 2Y (TU)", "ust_5y_fut": "UST 5Y (FV)",
    "ust_10y_fut": "UST 10Y (TY)", "ust_30y_fut": "UST 30Y (US)",
    "es_fut_front": "S&P 500 (ES)", "nq_fut_front": "Nasdaq (NQ)",
    "stoxx50_fut_front": "EuroStoxx 50", "nikkei_fut_front": "Nikkei 225",
    "eurusd_spot": "EURUSD", "usdjpy_spot": "USDJPY", "gbpusd_spot": "GBPUSD",
    "audusd_spot": "AUDUSD", "nzdusd_spot": "NZDUSD", "usdcad_spot": "USDCAD",
}

INST_BUCKET = {}
for bucket, instruments in universe.items():
    for inst in instruments:
        INST_BUCKET[inst] = bucket

def resolve_ticker(logical_name, ticker_map):
    for group in ticker_map.values():
        if isinstance(group, dict) and logical_name in group:
            return group[logical_name]
    return None

resolved = {}
missing = []
for bucket, instruments in universe.items():
    for inst in instruments:
        bbg = resolve_ticker(inst, tickers)
        if bbg:
            resolved[inst] = bbg
        else:
            missing.append(inst)

print(f"Universe: {sum(len(v) for v in universe.values())} instruments across {len(universe)} buckets")
print(f"Resolved: {len(resolved)} | Missing: {len(missing)}")
if missing:
    print(f"\n  Missing tickers (will be skipped): {missing}")

print("\nBucket breakdown:")
for bucket, instruments in universe.items():
    n_resolved = sum(1 for i in instruments if i in resolved)
    print(f"  {bucket:15s}: {n_resolved}/{len(instruments)} resolved")

## Data Pipeline (BQL)

Fetch daily `PX_LAST` for the full multi-asset universe.
Uses corrected BQL syntax — `df.set_index('DATE')`.
Instruments with < 252+30 days of history are dropped.

**Fix 3:** After panel construction, each instrument is masked to its
valid date range only. Pre-listing NaN periods are NOT forward-filled.

In [ ]:
class BQuantDataLoader:
    """Fetch historical prices via Bloomberg BQL."""

    def __init__(self, ticker_map: dict) -> None:
        self._tickers = ticker_map
        self._bq = bql.Service()

    def resolve(self, logical_name: str) -> str:
        for group in self._tickers.values():
            if isinstance(group, dict) and logical_name in group:
                return group[logical_name]
        raise KeyError(f"{logical_name} not in tickers.yaml")

    def get_history(self, logical_name: str, start: str, end: str,
                    field: str = "PX_LAST") -> pd.Series:
        bbg = self.resolve(logical_name)
        request = bql.Request(
            bbg,
            {field: self._bq.data.px_last(
                dates=self._bq.func.range(start, end)
            )},
        )
        try:
            response = self._bq.execute(request)
            df = response[0].df()
            if df.empty:
                return pd.Series(dtype=float, name=logical_name)
            df_fixed = df.set_index('DATE')
            series = df_fixed[field]
            series.index = pd.to_datetime(series.index, errors='coerce')
            series = series.dropna()
            series = series[~series.index.duplicated(keep='last')]
            series = series.sort_index().astype(float)
            series.name = logical_name
            series.index.name = "date"
            return series
        except Exception as exc:
            print(f"  ! {logical_name} ({bbg}): {exc}")
            return pd.Series(dtype=float, name=logical_name)


loader = BQuantDataLoader(tickers)
prices_raw = {}
fetch_status = []

for inst in resolved:
    label = LABELS.get(inst, inst)
    bucket = INST_BUCKET[inst]
    print(f"  {label:22s}", end=" ")
    s = loader.get_history(inst, IS_START, IS_END)
    n_obs = len(s)
    if n_obs > 0:
        prices_raw[inst] = s
        print(f"OK  {n_obs:>5d} obs  [{s.index[0]:%Y-%m-%d} -> {s.index[-1]:%Y-%m-%d}]")
    else:
        print("MISSING")
    fetch_status.append({"instrument": inst, "label": label, "bucket": bucket,
                         "obs": n_obs, "status": "OK" if n_obs > 0 else "MISSING"})

# Build aligned panel — DO NOT ffill across pre-listing gaps (Fix 3)
prices_df = pd.DataFrame(prices_raw).sort_index()
prices_df = prices_df[prices_df.index.notna()]

# ── FIX 3: Record first valid date per instrument ────────────────
FIRST_VALID_DATE = {}
for col in prices_df.columns:
    fvd = prices_df[col].first_valid_index()
    FIRST_VALID_DATE[col] = fvd

# Only ffill WITHIN each instrument's valid range (not before first valid)
for col in prices_df.columns:
    fvd = FIRST_VALID_DATE[col]
    if fvd is not None:
        # Set everything before first valid to NaN (don't ffill into it)
        prices_df.loc[:fvd, col] = prices_df.loc[:fvd, col]  # keep as-is
        # ffill only after first valid date
        valid_mask = prices_df.index >= fvd
        prices_df.loc[valid_mask, col] = prices_df.loc[valid_mask, col].ffill()

# Drop instruments with insufficient history
MIN_OBS = LOOKBACK_DAYS + 30
sufficient = prices_df.count() >= MIN_OBS
dropped_insts = prices_df.columns[~sufficient].tolist()
if dropped_insts:
    print(f"\nDropped (< {MIN_OBS} obs): {[LABELS.get(i,i) for i in dropped_insts]}")
prices_df = prices_df[prices_df.columns[sufficient]]

active_instruments = list(prices_df.columns)
active_buckets = {}
for inst in active_instruments:
    b = INST_BUCKET.get(inst)
    if b:
        active_buckets.setdefault(b, []).append(inst)

print(f"\nFinal panel: {prices_df.shape[0]} days x {prices_df.shape[1]} instruments")

# ── FIX 3: Print first valid dates ───────────────────────────────
print("\n" + "=" * 65)
print("  INSTRUMENT ACTIVE DATE RANGES (FIX 3)")
print("=" * 65)
for col in prices_df.columns:
    fvd = FIRST_VALID_DATE.get(col)
    n_pre = 0
    if fvd is not None:
        n_pre = (prices_df.index < fvd).sum()
    label = LABELS.get(col, col)
    print(f"  {label:22s}  active from: {str(fvd)[:10] if fvd else 'N/A':12s}  "
          f"({n_pre} days zeroed pre-listing)")

# Breadth report
print("\n" + "=" * 65)
print("  BREADTH REPORT")
print("=" * 65)
status_df = pd.DataFrame(fetch_status)
for bucket in universe:
    bdf = status_df[status_df["bucket"] == bucket]
    n_total = len(bdf)
    n_ok = (bdf["status"] == "OK").sum()
    n_sufficient = sum(1 for i in universe[bucket] if i in prices_df.columns)
    print(f"  {bucket:15s}: {n_total} defined | {n_ok} fetched | {n_sufficient} with >= {MIN_OBS}d history")

## DEBUG 1 — Raw Data Integrity Checks

Assert index is sorted and unique. Report NaN counts, % missing,
first/last valid dates per instrument.

In [ ]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

assert prices_df.index.is_monotonic_increasing, "Index not sorted!"
assert not prices_df.index.duplicated().any(), "Duplicate dates in index!"
print("Index check: sorted=True, unique=True")

coverage_rows = []
for col in prices_df.columns:
    s = prices_df[col]
    n_total = len(s)
    n_nan = s.isna().sum()
    first_valid = s.first_valid_index()
    last_valid = s.last_valid_index()
    coverage_rows.append({
        "instrument": col, "label": LABELS.get(col, col),
        "bucket": INST_BUCKET.get(col, "?"),
        "n_obs": n_total, "n_nan": n_nan,
        "pct_missing": n_nan / n_total * 100,
        "first_valid": str(first_valid)[:10] if first_valid else "N/A",
        "last_valid": str(last_valid)[:10] if last_valid else "N/A",
    })

coverage_df = pd.DataFrame(coverage_rows).sort_values("pct_missing", ascending=False)
coverage_df.to_csv(output_dir / "debug_coverage.csv", index=False)

print("\n" + "=" * 70)
print("  DATA COVERAGE (sorted by missingness)")
print("=" * 70)
for _, row in coverage_df.iterrows():
    flag = " *** " if row["pct_missing"] > 5 else "     "
    print(f"{flag}{row['label']:22s} [{row['bucket']:12s}]  "
          f"miss={row['pct_missing']:5.1f}%  "
          f"range={row['first_valid']} -> {row['last_valid']}")

print(f"\nExported: debug_coverage.csv")

## DEBUG 1B — Price vs Yield / Units Audit

Classify each series heuristically to detect instruments that look like
yields (bounded 0–100, small absolute moves) vs futures prices.
Critical for SOFR futures and UST futures which quote as `100 - rate`.

In [ ]:
def classify_series(s: pd.Series) -> dict:
    s = s.dropna()
    if len(s) < 50:
        return {"last": np.nan, "median": np.nan, "min": np.nan, "max": np.nan,
                "diff_1d_med": np.nan, "pct_1d_med": np.nan,
                "bounded_0_100": False, "looks_yield": False, "looks_price": False}
    diff_abs = s.diff().abs()
    pct_abs = s.pct_change().abs()
    bounded = ((s >= 0) & (s <= 100)).mean() >= 0.95
    med = s.median()
    looks_yield = bounded and med < 25 and diff_abs.median() < 0.10
    looks_price = pct_abs.median() < 0.02 and med > 10
    return {"last": s.iloc[-1], "median": med, "min": s.min(), "max": s.max(),
            "diff_1d_med": diff_abs.median(), "pct_1d_med": pct_abs.median(),
            "bounded_0_100": bounded, "looks_yield": looks_yield, "looks_price": looks_price}

MUST_BE_PRICE = set(active_instruments)

audit_rows = []
for col in prices_df.columns:
    info = classify_series(prices_df[col])
    info["instrument"] = col
    info["label"] = LABELS.get(col, col)
    info["warning"] = (col in MUST_BE_PRICE and info["looks_yield"])
    audit_rows.append(info)

units_audit_df = pd.DataFrame(audit_rows)
units_audit_df.to_csv(output_dir / "debug_units_audit.csv", index=False)

flagged = units_audit_df[units_audit_df["warning"]]
if len(flagged) > 0:
    print("!" * 70)
    print("  CRITICAL: Must-be-price instruments flagged as YIELD-LIKE")
    print("!" * 70)
    for _, row in flagged.iterrows():
        print(f"  {row['label']:22s}  median={row['median']:.2f}  "
              f"diff_1d={row['diff_1d_med']:.4f}  bounded={row['bounded_0_100']}")
else:
    print("Units audit: no critical warnings.")

print("\n" + "=" * 70)
print("  UNITS AUDIT SUMMARY")
print("=" * 70)
for _, row in units_audit_df.iterrows():
    tag = "YIELD?" if row["looks_yield"] else "PRICE " if row["looks_price"] else "????? "
    warn = " <<<WARN" if row["warning"] else ""
    print(f"  {tag} {row['label']:22s}  med={row['median']:>10.2f}  "
          f"pct_1d={row['pct_1d_med']:.4f}  diff_1d={row['diff_1d_med']:.4f}{warn}")

print(f"\nExported: debug_units_audit.csv")

## DEBUG 2 — Price and Return Sanity (per instrument)

Compute return statistics and flag instruments with extreme daily moves.

In [ ]:
ret_pct = prices_df.pct_change()

ret_stats_rows = []
for col in prices_df.columns:
    r = ret_pct[col].dropna()
    if len(r) == 0:
        continue
    ret_stats_rows.append({
        "instrument": col, "label": LABELS.get(col, col),
        "bucket": INST_BUCKET.get(col, "?"),
        "p01": r.quantile(0.01), "p50": r.quantile(0.50), "p99": r.quantile(0.99),
        "min": r.min(), "max": r.max(),
        "count_gt_1pct": (r.abs() > 0.01).sum(),
        "count_gt_2pct": (r.abs() > 0.02).sum(),
        "count_gt_5pct": (r.abs() > 0.05).sum(),
    })

ret_stats_df = pd.DataFrame(ret_stats_rows)
ret_stats_df.to_csv(output_dir / "debug_return_stats.csv", index=False)

print("=" * 75)
print("  RETURN STATISTICS (daily pct_change)")
print("=" * 75)
for _, row in ret_stats_df.iterrows():
    warn = ""
    if row["bucket"] == "rates" and row["count_gt_1pct"] > 50:
        warn = " *** HIGH FREQ >1% MOVES (rates) ***"
    print(f"  {row['label']:22s}  p01={row['p01']:+.3f}  p99={row['p99']:+.3f}  "
          f">1%={row['count_gt_1pct']:4d}  >5%={row['count_gt_5pct']:3d}{warn}")

print(f"\nExported: debug_return_stats.csv")

## DEBUG 3 — Outlier Dates (Top Shocks)

For each instrument, list the top 10 absolute daily return dates.

In [ ]:
top_shocks = {}
shock_export = []

for col in prices_df.columns:
    r = ret_pct[col].dropna().abs()
    top10 = r.nlargest(10)
    top_shocks[col] = top10
    for dt, val in top10.items():
        shock_export.append({
            "instrument": col, "label": LABELS.get(col, col),
            "date": str(dt)[:10], "abs_return": val,
            "price": prices_df.loc[dt, col] if dt in prices_df.index else np.nan,
        })

shock_df = pd.DataFrame(shock_export)
shock_df.to_csv(output_dir / "debug_top_shocks.csv", index=False)

print("=" * 70)
print("  TOP 10 SHOCKS — RATES FUTURES")
print("=" * 70)
rates_keys = [c for c in prices_df.columns if INST_BUCKET.get(c) == "rates"]
for col in rates_keys[:4]:
    label = LABELS.get(col, col)
    print(f"\n  {label}:")
    for dt, val in top_shocks[col].items():
        price = prices_df.loc[dt, col] if dt in prices_df.index else np.nan
        print(f"    {str(dt)[:10]}  |ret|={val:.4f} ({val*100:.2f}%)  price={price:.4f}")

print(f"\nExported: debug_top_shocks.csv")

## DEBUG 4 — Roll / Stitch Artifact Detection

Flag dates where `|diff| / rolling_std(20)` > 8 — likely roll or data stitch.

In [ ]:
roll_flags = []
for col in prices_df.columns:
    d = prices_df[col].diff()
    rs = d.rolling(20, min_periods=10).std().replace(0, np.nan)
    standardized = (d.abs() / rs).dropna()
    flagged = standardized[standardized > 8]
    for dt, val in flagged.items():
        roll_flags.append({
            "instrument": col, "label": LABELS.get(col, col),
            "date": str(dt)[:10], "standardized_jump": val,
            "price": prices_df.loc[dt, col], "diff": d.loc[dt],
        })

roll_df = pd.DataFrame(roll_flags)
roll_df.to_csv(output_dir / "debug_roll_flags.csv", index=False)

print("=" * 70)
print("  ROLL / STITCH ARTIFACT FLAGS (|diff|/rolling_std > 8)")
print("=" * 70)
if len(roll_df) == 0:
    print("  None detected.")
else:
    counts = roll_df.groupby("label").size().sort_values(ascending=False)
    for label, cnt in counts.items():
        print(f"  {label:22s}  {cnt} flagged dates")

print(f"\nExported: debug_roll_flags.csv")

## DEBUG 5 — EWMA Vol Calculation Verification

Explicitly recompute EWMA vol and check for double-annualisation.
If median annualised vol > 80%, something is wrong.

In [ ]:
lambda_ = VOL_LAMBDA
alpha_ = 1 - lambda_

ret_raw = prices_df.pct_change().fillna(0.0)

# ── FIX 3: Zero out returns before first valid date ──────────────
for col in ret_raw.columns:
    fvd = FIRST_VALID_DATE.get(col)
    if fvd is not None:
        ret_raw.loc[ret_raw.index < fvd, col] = 0.0

# ── FIX 5: Winsorize returns at ±5 z-scores ─────────────────────
rolling_std_20 = ret_raw.rolling(20, min_periods=5).std()
threshold = WINSOR_Z_THRESHOLD * rolling_std_20
ret_clean = ret_raw.clip(-threshold, threshold)

# Count winsorized observations
n_winsorized = (ret_raw != ret_clean).sum()
print("=" * 70)
print(f"  RETURN WINSORIZATION (FIX 5, z-threshold={WINSOR_Z_THRESHOLD})")
print("=" * 70)
for col in ret_clean.columns:
    n_w = n_winsorized[col]
    if n_w > 0:
        label = LABELS.get(col, col)
        print(f"  {label:22s}  {n_w} observations winsorized")
if n_winsorized.sum() == 0:
    print("  No observations winsorized.")

# Recompute vol from CLEAN returns
var_ewma = ret_clean.pow(2).ewm(alpha=alpha_, adjust=False).mean()
vol_daily_check = np.sqrt(var_ewma)
vol_ann_check = vol_daily_check * sqrt(252)

print("\n" + "=" * 70)
print(f"  EWMA VOL VERIFICATION  (lambda={lambda_}, alpha={alpha_})")
print("=" * 70)

vol_summary_rows = []
for col in vol_ann_check.columns:
    v = vol_ann_check[col].dropna()
    vol_summary_rows.append({
        "instrument": col, "label": LABELS.get(col, col),
        "bucket": INST_BUCKET.get(col, "?"),
        "vol_ann_mean": v.mean(), "vol_ann_median": v.median(),
        "vol_ann_min": v.min(), "vol_ann_max": v.max(),
    })
    print(f"  {LABELS.get(col,col):22s}  mean={v.mean():.1%}  "
          f"med={v.median():.1%}  range=[{v.min():.1%}, {v.max():.1%}]")

vol_summary_df = pd.DataFrame(vol_summary_rows)
vol_summary_df.to_csv(output_dir / "debug_vol_summary.csv", index=False)

median_mean_vol = vol_summary_df["vol_ann_mean"].median()
if median_mean_vol > 0.80:
    print(f"\n  *** WARNING: median(mean vol_ann) = {median_mean_vol:.0%} > 80% ***")
else:
    print(f"\n  Vol check: median(mean vol_ann) = {median_mean_vol:.1%} — OK")

print(f"\nExported: debug_vol_summary.csv")

## Canonical TSMOM Signal

**Moskowitz, Ooi & Pedersen (2012):**
- Compute 12-month (252 trading day) return for each instrument
- Signal = `sign(return)`: +1 long, -1 short
- Resample to month-end, shift by 1 month (no lookahead)
- Forward-fill within each month

**Fix 3:** Signals zeroed before each instrument's first valid date.
**Fix 4:** Explicit shift verification included below.

In [ ]:
class CanonicalTSMOMStrategy:
    """Canonical TSMOM: sign(12M return), monthly rebalance."""

    def __init__(self, lookback_days: int = 252) -> None:
        self.lookback_days = lookback_days

    def compute_monthly_signal(self, prices_df: pd.DataFrame,
                               first_valid_dates: dict) -> pd.DataFrame:
        """Monthly-rebalanced sign(12M return) signal.

        Returns DataFrame of daily signals in {-1, 0, +1}.
        Fix 3: zero signals before first_valid_date per instrument.
        """
        r12 = prices_df.pct_change(self.lookback_days)
        sig_daily = np.sign(r12)
        # Avoid lookahead: use yesterday's signal
        sig_daily = sig_daily.shift(1)
        # Month-end snapshot, shifted by 1 month
        signal_monthly = sig_daily.resample("M").last()
        signal_monthly = signal_monthly.shift(1)
        # Forward-fill to daily
        signal_daily = signal_monthly.reindex(prices_df.index).ffill().fillna(0.0)

        # FIX 3: Zero signals before first valid date
        for col in signal_daily.columns:
            fvd = first_valid_dates.get(col)
            if fvd is not None:
                # Need lookback + 1 month buffer before signal is valid
                min_signal_date = fvd + pd.Timedelta(days=self.lookback_days + 35)
                signal_daily.loc[signal_daily.index < min_signal_date, col] = 0.0

        return signal_daily


tsmom = CanonicalTSMOMStrategy(lookback_days=LOOKBACK_DAYS)
signals = tsmom.compute_monthly_signal(prices_df, FIRST_VALID_DATE)

print(f"Signal matrix: {signals.shape[0]} days x {signals.shape[1]} instruments")

# ── Signal distribution ──────────────────────────────────────────
print("\n" + "=" * 65)
print("  SIGNAL DISTRIBUTION (% of days)")
print("=" * 65)
for col in signals.columns:
    s = signals[col]
    n = len(s)
    pct_long  = (s ==  1).sum() / n * 100
    pct_short = (s == -1).sum() / n * 100
    pct_flat  = (s ==  0).sum() / n * 100
    label = LABELS.get(col, col)
    print(f"  {label:22s}  Long={pct_long:5.1f}%  Short={pct_short:5.1f}%  Flat={pct_flat:5.1f}%")

# ── Holding period ───────────────────────────────────────────────
print("\n" + "=" * 65)
print("  AVERAGE HOLDING PERIOD")
print("=" * 65)
for col in signals.columns:
    changes = (signals[col].diff().abs() > 0).sum()
    avg_hold = len(signals[col]) / max(changes, 1)
    label = LABELS.get(col, col)
    print(f"  {label:22s}  {avg_hold:.0f} days  ({avg_hold/21:.1f} months)")

unique_vals = set()
for col in signals.columns:
    unique_vals.update(signals[col].dropna().unique())
assert unique_vals <= {-1.0, 0.0, 1.0}, "Signal values outside {-1, 0, +1}!"
print(f"\nSignal check passed: strictly in {{-1, 0, +1}}")

## Monthly Frozen Vol Scaling + Hard Absolute Weight Caps

**EWMA volatility** (lambda=0.94) on CLEANED returns (Fix 5: winsorized).
Month-end snapshot, shifted by 1 month, forward-filled intra-month.

**Fix 1:** After relative leverage cap, apply HARD ABSOLUTE CAP per bucket:
- Commodities: 1.0 (energy: 0.5)
- Equities: 0.8
- FX: 0.8
- Rates: 0.5

**Fix 3:** Weights zeroed before each instrument's first valid date.

In [ ]:
def ex_ante_vol_ewma(returns: pd.DataFrame, lam: float = 0.94) -> pd.DataFrame:
    """EWMA annualised volatility (RiskMetrics style)."""
    alpha = 1 - lam
    ewma_var = returns.pow(2).ewm(alpha=alpha, adjust=False).mean()
    vol_daily = np.sqrt(ewma_var)
    vol_ann = vol_daily * sqrt(252)
    return vol_ann


# Use CLEAN returns (winsorized in cell 19)
ret = ret_clean.copy()
vol_ann = ex_ante_vol_ewma(ret, lam=VOL_LAMBDA)

# Month-end vol snapshot, shifted by 1 month
vol_monthly = vol_ann.resample("M").last().shift(1)
vol_frozen_daily = vol_monthly.reindex(prices_df.index).ffill()
vol_frozen_daily = vol_frozen_daily.replace(0, np.nan)

# ── Raw weights ──────────────────────────────────────────────────
raw_w = signals * (TARGET_VOL / vol_frozen_daily)

# ── Relative leverage cap ────────────────────────────────────────
w_cap = LEV_CAP * (TARGET_VOL / vol_frozen_daily)
weights_rel_capped = raw_w.clip(-w_cap, w_cap).fillna(0.0)

# ── FIX 1: Hard absolute weight cap per bucket ──────────────────
print("=" * 65)
print("  FIX 1 — HARD ABSOLUTE WEIGHT CAP")
print("=" * 65)
print("\n  Max |weight| BEFORE hard cap:")
weights = weights_rel_capped.copy()
for col in weights.columns:
    max_raw = weights[col].abs().max()
    label = LABELS.get(col, col)
    bucket = INST_BUCKET.get(col, "?")
    print(f"    {label:22s} [{bucket:12s}]  max|w|={max_raw:.3f}")

# Apply hard cap
for col in weights.columns:
    bucket = INST_BUCKET.get(col, "commodities")
    cap = ABS_WEIGHT_CAP_BY_BUCKET.get(bucket, 1.0)
    # Per-instrument override
    if col in ABS_WEIGHT_CAP_BY_INSTRUMENT:
        cap = ABS_WEIGHT_CAP_BY_INSTRUMENT[col]
    weights[col] = weights[col].clip(-cap, cap)

print("\n  Max |weight| AFTER hard cap:")
for col in weights.columns:
    max_post = weights[col].abs().max()
    label = LABELS.get(col, col)
    bucket = INST_BUCKET.get(col, "?")
    cap_used = ABS_WEIGHT_CAP_BY_INSTRUMENT.get(col,
                ABS_WEIGHT_CAP_BY_BUCKET.get(bucket, 1.0))
    print(f"    {label:22s} [{bucket:12s}]  max|w|={max_post:.3f}  (cap={cap_used})")

# ASSERTION: no weight exceeds 1.0
max_abs_all = weights.abs().max().max()
assert max_abs_all <= 1.0 + 1e-10, f"FIX 1 FAILED: max|w|={max_abs_all:.4f} > 1.0"
print(f"\n  ASSERTION PASSED: max(abs(weights)) = {max_abs_all:.4f} <= 1.0")

# ── FIX 3: Zero weights before first valid date ─────────────────
for col in weights.columns:
    fvd = FIRST_VALID_DATE.get(col)
    if fvd is not None:
        min_date = fvd + pd.Timedelta(days=LOOKBACK_DAYS + 35)
        weights.loc[weights.index < min_date, col] = 0.0

# ── FIX 3 assertion: SOFR weights zero before 2018-05-04 ────────
sofr_check_date = pd.Timestamp("2018-05-04")
sofr_insts = [c for c in weights.columns if c.startswith("sofr_")]
for inst in sofr_insts:
    pre_sofr = weights.loc[weights.index < sofr_check_date, inst]
    n_nonzero = (pre_sofr.abs() > 1e-10).sum()
    label = LABELS.get(inst, inst)
    if n_nonzero == 0:
        print(f"  FIX 3 OK: {label} has 0 non-zero weights before 2018-05-04")
    else:
        print(f"  FIX 3 FAIL: {label} has {n_nonzero} non-zero weights before 2018-05-04!")

# Weight change frequency
print("\n" + "=" * 65)
print("  WEIGHT CHANGE FREQUENCY (expect ~12/year)")
print("=" * 65)
for col in weights.columns:
    changes = (weights[col].diff().abs() > 1e-10).sum()
    label = LABELS.get(col, col)
    print(f"  {label:22s}  {changes:4d} changes")

## Sector Risk Budgets (Commodity-Focused)

Scale weights so the momentum sleeve stays commodity-focused:
- **Commodities:** 70% of portfolio risk
- **Diversifiers** (rates + equities + FX): 30% of portfolio risk

Risk per bucket = `sum(|w_i| * vol_i)` within bucket.
Computed monthly (month-end), shifted, forward-filled.

In [ ]:
def compute_bucket_scaling(
    weights: pd.DataFrame, vol_frozen: pd.DataFrame,
    inst_bucket: dict,
    commodity_budget: float = 0.70, diversifier_budget: float = 0.30,
) -> pd.DataFrame:
    """Compute monthly bucket-level scaling factors."""
    commodity_insts = [i for i in weights.columns if inst_bucket.get(i) == "commodities"]
    diversifier_insts = [i for i in weights.columns if inst_bucket.get(i) != "commodities"]

    risk_comm = (weights[commodity_insts].abs() * vol_frozen[commodity_insts]).sum(axis=1) if commodity_insts else pd.Series(0.0, index=weights.index)
    risk_div  = (weights[diversifier_insts].abs() * vol_frozen[diversifier_insts]).sum(axis=1) if diversifier_insts else pd.Series(0.0, index=weights.index)
    total_risk = risk_comm + risk_div

    target_comm = total_risk * commodity_budget
    target_div  = total_risk * diversifier_budget

    scale_comm = (target_comm / risk_comm.replace(0, np.nan)).fillna(1.0)
    scale_div  = (target_div / risk_div.replace(0, np.nan)).fillna(1.0)

    # Freeze monthly
    scale_comm_m = scale_comm.resample("M").last().shift(1)
    scale_div_m  = scale_div.resample("M").last().shift(1)
    scale_comm_d = scale_comm_m.reindex(weights.index).ffill().fillna(1.0)
    scale_div_d  = scale_div_m.reindex(weights.index).ffill().fillna(1.0)

    scaling = pd.DataFrame(1.0, index=weights.index, columns=weights.columns)
    for inst in commodity_insts:
        scaling[inst] = scale_comm_d
    for inst in diversifier_insts:
        scaling[inst] = scale_div_d
    return scaling


bucket_scaling = compute_bucket_scaling(
    weights, vol_frozen_daily, INST_BUCKET,
    commodity_budget=COMMODITY_RISK_BUDGET,
    diversifier_budget=DIVERSIFIER_RISK_BUDGET,
)

weights_scaled = weights * bucket_scaling

# Re-apply hard absolute cap after bucket scaling (safety net)
for col in weights_scaled.columns:
    bucket = INST_BUCKET.get(col, "commodities")
    cap = ABS_WEIGHT_CAP_BY_INSTRUMENT.get(col,
          ABS_WEIGHT_CAP_BY_BUCKET.get(bucket, 1.0))
    weights_scaled[col] = weights_scaled[col].clip(-cap, cap)

# Diagnostics
commodity_insts = [i for i in weights_scaled.columns if INST_BUCKET.get(i) == "commodities"]
diversifier_insts = [i for i in weights_scaled.columns if INST_BUCKET.get(i) != "commodities"]

risk_comm_post = (weights_scaled[commodity_insts].abs() * vol_frozen_daily[commodity_insts]).sum(axis=1)
risk_div_post  = (weights_scaled[diversifier_insts].abs() * vol_frozen_daily[diversifier_insts]).sum(axis=1)
total_risk_post = risk_comm_post + risk_div_post

bucket_risk_shares = pd.DataFrame({
    "commodities_pct": (risk_comm_post / total_risk_post.replace(0, np.nan) * 100).fillna(0),
    "diversifiers_pct": (risk_div_post / total_risk_post.replace(0, np.nan) * 100).fillna(0),
})

avg_comm_share = bucket_risk_shares["commodities_pct"].mean()
avg_div_share  = bucket_risk_shares["diversifiers_pct"].mean()

print("=" * 65)
print("  REALISED RISK SHARE BY BUCKET (after scaling)")
print("=" * 65)
print(f"  Commodities  : {avg_comm_share:.1f}%  (target: {COMMODITY_RISK_BUDGET*100:.0f}%)")
print(f"  Diversifiers : {avg_div_share:.1f}%  (target: {DIVERSIFIER_RISK_BUDGET*100:.0f}%)")

for bucket in universe:
    insts = [i for i in weights_scaled.columns if INST_BUCKET.get(i) == bucket]
    if insts:
        risk_b = (weights_scaled[insts].abs() * vol_frozen_daily[insts]).sum(axis=1)
        share = (risk_b / total_risk_post.replace(0, np.nan) * 100).mean()
        print(f"    {bucket:15s}: {share:.1f}%  ({len(insts)} instruments)")

## Portfolio Vol Targeting Overlay

After bucket scaling, apply portfolio-level vol overlay targeting 10% annual:
- Rolling 60-day realised portfolio vol
- Scale = target / realised, capped to **[0.05, 2.0]** (Fix 2: lowered floor)
- Computed monthly (month-end), shifted by 1, forward-filled daily

In [ ]:
# Pre-overlay portfolio returns
port_ret_pre = (weights_scaled.shift(1) * ret).sum(axis=1)

# Rolling portfolio vol
port_vol_rolling = port_ret_pre.rolling(PORTFOLIO_VOL_LOOKBACK, min_periods=20).std() * sqrt(252)

# Scale factor (FIX 2: floor=0.05 instead of 0.5)
port_scale = PORTFOLIO_VOL_TARGET / port_vol_rolling.replace(0, np.nan)
port_scale = port_scale.clip(PORTFOLIO_SCALE_BOUNDS[0], PORTFOLIO_SCALE_BOUNDS[1]).fillna(1.0)

# Freeze monthly
port_scale_monthly = port_scale.resample("M").last().shift(1)
port_scale_daily = port_scale_monthly.reindex(prices_df.index).ffill().fillna(1.0)

# Final weights
final_weights = weights_scaled.multiply(port_scale_daily, axis=0)

# Re-apply hard abs cap one final time (after overlay)
for col in final_weights.columns:
    bucket = INST_BUCKET.get(col, "commodities")
    cap = ABS_WEIGHT_CAP_BY_INSTRUMENT.get(col,
          ABS_WEIGHT_CAP_BY_BUCKET.get(bucket, 1.0))
    final_weights[col] = final_weights[col].clip(-cap, cap)

# ── FIX 2: Print overlay diagnostics ─────────────────────────────
print("=" * 65)
print("  PORTFOLIO VOL OVERLAY (FIX 2: floor=0.05)")
print("=" * 65)
print(f"  Pre-overlay avg vol : {port_vol_rolling.dropna().mean():.1%}")
print(f"  Overlay scale range : [{port_scale_daily.min():.3f}, {port_scale_daily.max():.3f}]")
print(f"  Overlay scale mean  : {port_scale_daily.mean():.3f}")

# Post-overlay vol check
port_ret_post = (final_weights.shift(1) * ret).sum(axis=1)
port_vol_post = port_ret_post.rolling(60, min_periods=20).std() * sqrt(252)
mean_post_vol = port_vol_post.dropna().mean()

print(f"  Post-overlay avg vol: {mean_post_vol:.1%}  (target: {PORTFOLIO_VOL_TARGET:.0%})")
print(f"  Post-overlay range  : [{port_vol_post.dropna().min():.1%}, {port_vol_post.dropna().max():.1%}]")

# FIX 2 ASSERTION
if 0.06 <= mean_post_vol <= 0.14:
    print(f"\n  ASSERTION PASSED: mean post-overlay vol {mean_post_vol:.1%} in [6%, 14%]")
else:
    print(f"\n  ASSERTION WARNING: mean post-overlay vol {mean_post_vol:.1%} outside [6%, 14%]")
    print(f"  Suggested diagnostic: check if bucket scaling inflates weights excessively")

# Confirm monthly freeze
total_changes = 0
for col in final_weights.columns:
    changes = (final_weights[col].diff().abs() > 1e-10).sum()
    total_changes += changes
avg_changes = total_changes / len(final_weights.columns)
print(f"\n  Avg weight changes per instrument: {avg_changes:.0f}")

## Backtest

- `port_ret_gross = sum(final_weight(t-1) * return(t))`
- Turnover from final weight changes; TC = turnover * bp/10000
- **No division by n_active** — weights already vol-scaled and budget-constrained

In [ ]:
# Portfolio returns
port_ret_gross = (final_weights.shift(1) * ret).sum(axis=1)
turnover_daily = (final_weights - final_weights.shift(1)).abs().sum(axis=1).fillna(0.0)
tc_daily = turnover_daily * (TC_BP / 10_000)
port_ret_net = port_ret_gross - tc_daily
port_equity = 1_000_000 * (1 + port_ret_net).cumprod()

# FINAL ASSERTION 5: No NaN or inf in portfolio returns
n_bad = (~np.isfinite(port_ret_net)).sum()
assert n_bad == 0, f"FINAL ASSERTION 5 FAILED: {n_bad} non-finite portfolio returns"
print(f"ASSERTION 5 PASSED: 0 non-finite values in portfolio returns")

# Per-instrument metrics
inst_equity = {}
inst_metrics = []
for col in final_weights.columns:
    r_inst = (final_weights[col].shift(1) * ret[col])
    to_inst = final_weights[col].diff().abs().fillna(0.0)
    tc_inst = to_inst * (TC_BP / 10_000)
    r_net = r_inst - tc_inst
    eq = 1_000_000 * (1 + r_net).cumprod()
    inst_equity[col] = eq

    n_years = len(r_net) / 252
    total = (1 + r_net).prod()
    ann_ret = total ** (1 / max(n_years, 0.01)) - 1
    ann_vol = r_net.std() * sqrt(252)
    sharpe = ann_ret / ann_vol if ann_vol > 1e-10 else 0.0
    rm = eq.cummax(); dd = (eq - rm) / rm
    max_dd = float(-dd.min()) if len(dd) > 0 else 0.0
    calmar = ann_ret / max_dd if max_dd > 1e-10 else 0.0
    hit = float((r_net > 0).sum() / len(r_net)) if len(r_net) > 0 else 0.0
    ann_to = to_inst.sum() / max(n_years, 0.01)

    inst_metrics.append({
        "Instrument": LABELS.get(col, col), "Bucket": INST_BUCKET.get(col, "?"),
        "Ann. Return": ann_ret, "Ann. Vol": ann_vol,
        "Sharpe": sharpe, "Max DD": max_dd,
        "Calmar": calmar, "Hit Rate": hit, "Ann. Turnover": ann_to,
    })

# Portfolio-level metrics
r = port_ret_net; eq = port_equity
n_years = len(r) / 252
total = (1 + r).prod()
ann_ret = total ** (1 / max(n_years, 0.01)) - 1
ann_vol = r.std() * sqrt(252)
sharpe = ann_ret / ann_vol if ann_vol > 1e-10 else 0.0
rm = eq.cummax(); dd = (eq - rm) / rm
max_dd = float(-dd.min())
calmar = ann_ret / max_dd if max_dd > 1e-10 else 0.0
hit = float((r > 0).sum() / len(r))
ann_to = turnover_daily.sum() / max(n_years, 0.01)

port_metrics = {
    "Instrument": "PORTFOLIO", "Bucket": "all",
    "Ann. Return": ann_ret, "Ann. Vol": ann_vol,
    "Sharpe": sharpe, "Max DD": max_dd,
    "Calmar": calmar, "Hit Rate": hit, "Ann. Turnover": ann_to,
}

metrics_df = pd.DataFrame(inst_metrics + [port_metrics]).set_index("Instrument")

fmt_df = metrics_df.copy()
for c in ["Ann. Return", "Ann. Vol", "Max DD", "Hit Rate"]:
    fmt_df[c] = fmt_df[c].map("{:.1%}".format)
fmt_df["Sharpe"] = fmt_df["Sharpe"].map("{:.2f}".format)
fmt_df["Calmar"] = fmt_df["Calmar"].map("{:.2f}".format)
fmt_df["Ann. Turnover"] = fmt_df["Ann. Turnover"].map("{:.1f}x".format)

print("\n" + "=" * 75)
print("  CANONICAL TSMOM — METRICS (IS 2015-2022, POST-FIX)")
print("=" * 75)
display(fmt_df)

## DEBUG 6 — Weight & Exposure Sanity

Check max weights (should be <= 1.0 after Fix 1), gross/net exposure,
and bucket-level breakdowns.

In [ ]:
w_summary_rows = []
for col in final_weights.columns:
    w = final_weights[col]
    w_summary_rows.append({
        "instrument": col, "label": LABELS.get(col, col),
        "bucket": INST_BUCKET.get(col, "?"),
        "max_abs_weight": w.abs().max(),
        "mean_abs_weight": w.abs().mean(),
        "pct_gt_0.5": (w.abs() > 0.5).mean() * 100,
        "pct_gt_1.0": (w.abs() > 1.0).mean() * 100,
    })

w_summary_df = pd.DataFrame(w_summary_rows)
w_summary_df.to_csv(output_dir / "debug_weights_summary.csv", index=False)

gross_exp = final_weights.abs().sum(axis=1)
net_exp   = final_weights.sum(axis=1)

print("=" * 70)
print("  WEIGHT & EXPOSURE SUMMARY (post all fixes)")
print("=" * 70)
for _, row in w_summary_df.iterrows():
    warn = " <<<WARN" if row["max_abs_weight"] > 1.0 else ""
    print(f"  {row['label']:22s}  max|w|={row['max_abs_weight']:.3f}  "
          f">0.5={row['pct_gt_0.5']:.0f}%  >1.0={row['pct_gt_1.0']:.0f}%{warn}")

print(f"\n  Gross exposure: mean={gross_exp.mean():.2f}  max={gross_exp.max():.2f}")
print(f"  Net exposure:   mean={net_exp.mean():.2f}  range=[{net_exp.min():.2f}, {net_exp.max():.2f}]")

# FINAL ASSERTION 1
max_abs = w_summary_df["max_abs_weight"].max()
assert max_abs <= 1.0 + 1e-10, f"FINAL ASSERTION 1 FAILED: max|w|={max_abs:.4f}"
print(f"\n  FINAL ASSERTION 1 PASSED: max(abs(weights_final)) = {max_abs:.4f} <= 1.0")

print(f"\nExported: debug_weights_summary.csv")

## DEBUG 7 — Turnover Decomposition

Decompose turnover into month-boundary vs non-boundary contributions.
High non-boundary turnover indicates daily churn from overlay or bucket scaling.

In [ ]:
turnover_per_inst = (final_weights - final_weights.shift(1)).abs()
port_turnover_debug = turnover_per_inst.sum(axis=1).fillna(0.0)

periods = prices_df.index.to_period("M")
month_boundary = pd.Series(periods != periods.shift(1), index=prices_df.index)

to_boundary = port_turnover_debug[month_boundary]
to_non_boundary = port_turnover_debug[~month_boundary]

mean_boundary    = to_boundary.mean() if len(to_boundary) > 0 else 0
mean_nonboundary = to_non_boundary.mean() if len(to_non_boundary) > 0 else 0

print("=" * 70)
print("  TURNOVER DECOMPOSITION")
print("=" * 70)
print(f"  Total turnover days        : {(port_turnover_debug > 1e-10).sum()}")
print(f"  Month boundary days        : {month_boundary.sum()}")
print(f"  Mean turnover (boundary)   : {mean_boundary:.6f}")
print(f"  Mean turnover (non-bound.) : {mean_nonboundary:.6f}")
print(f"  Ratio (boundary/non)       : {mean_boundary / max(mean_nonboundary, 1e-10):.1f}x")

top20_turn = port_turnover_debug.nlargest(20)
n_top_boundary = sum(1 for dt in top20_turn.index if month_boundary.loc[dt])
print(f"\n  Top 20 turnover days: {n_top_boundary}/20 are month boundaries")

for dt, val in top20_turn.items():
    is_mb = "MB" if month_boundary.loc[dt] else "  "
    print(f"    {str(dt)[:10]}  {is_mb}  turnover={val:.6f}")

if mean_nonboundary > mean_boundary * 0.1:
    print("\n  *** WARNING: Daily churn detected ***")
else:
    print("\n  Overlay and bucket scaling confirmed monthly-frozen.")

turnover_export = pd.DataFrame({"turnover": port_turnover_debug, "month_boundary": month_boundary})
turnover_export.to_csv(output_dir / "debug_turnover.csv")
print(f"\nExported: debug_turnover.csv")

## DEBUG 8 — Control Tests (Signal vs Scaling)

Separate the contribution of the trend signal from vol scaling:
- **A) Baseline:** canonical TSMOM (sign + inv-vol + overlay)
- **B) Signal-only:** equal weights (no vol scaling)
- **C) Scaling-only:** always long (+1) with inv-vol scaling
- **D) Random-sign:** random +/-1 monthly with inv-vol scaling (seed=42)

In [ ]:
def quick_portfolio_metrics(ret_series, name=""):
    r = ret_series.dropna()
    eq = (1 + r).cumprod()
    n_years = len(r) / 252
    total = (1 + r).prod()
    ann_ret = total ** (1 / max(n_years, 0.01)) - 1
    ann_vol = r.std() * sqrt(252)
    sharpe = ann_ret / ann_vol if ann_vol > 1e-10 else 0.0
    rm = eq.cummax(); dd = (eq - rm) / rm
    max_dd = float(-dd.min()) if len(dd) > 0 else 0.0
    calmar = ann_ret / max_dd if max_dd > 1e-10 else 0.0
    return {"Variant": name, "Ann. Return": ann_ret, "Ann. Vol": ann_vol,
            "Sharpe": sharpe, "Max DD": max_dd, "Calmar": calmar}

ctrl_a = quick_portfolio_metrics(port_ret_net, "A) Baseline TSMOM")

n_inst = len(signals.columns)
w_equal = signals / n_inst
ret_b = (w_equal.shift(1) * ret).sum(axis=1)
ctrl_b = quick_portfolio_metrics(ret_b, "B) Signal-only (equal wt)")

always_long = pd.DataFrame(1.0, index=signals.index, columns=signals.columns)
w_long_raw = always_long * (TARGET_VOL / vol_frozen_daily)
w_long_cap = w_long_raw.clip(-LEV_CAP, LEV_CAP).fillna(0.0)
for col in w_long_cap.columns:
    bucket = INST_BUCKET.get(col, "commodities")
    cap = ABS_WEIGHT_CAP_BY_INSTRUMENT.get(col,
          ABS_WEIGHT_CAP_BY_BUCKET.get(bucket, 1.0))
    w_long_cap[col] = w_long_cap[col].clip(-cap, cap)
w_long_bucket = w_long_cap * bucket_scaling
w_long_final = w_long_bucket.multiply(port_scale_daily, axis=0)
ret_c = (w_long_final.shift(1) * ret).sum(axis=1)
ctrl_c = quick_portfolio_metrics(ret_c, "C) Scaling-only (always long)")

np.random.seed(42)
monthly_dates = signals.resample("M").last().index
random_signs = pd.DataFrame(
    np.random.choice([-1.0, 1.0], size=(len(monthly_dates), len(signals.columns))),
    index=monthly_dates, columns=signals.columns,
)
random_daily = random_signs.reindex(signals.index).ffill().fillna(0.0)
w_rand_raw = random_daily * (TARGET_VOL / vol_frozen_daily)
w_rand_cap = w_rand_raw.clip(-LEV_CAP, LEV_CAP).fillna(0.0)
for col in w_rand_cap.columns:
    bucket = INST_BUCKET.get(col, "commodities")
    cap = ABS_WEIGHT_CAP_BY_INSTRUMENT.get(col,
          ABS_WEIGHT_CAP_BY_BUCKET.get(bucket, 1.0))
    w_rand_cap[col] = w_rand_cap[col].clip(-cap, cap)
w_rand_bucket = w_rand_cap * bucket_scaling
w_rand_final = w_rand_bucket.multiply(port_scale_daily, axis=0)
ret_d = (w_rand_final.shift(1) * ret).sum(axis=1)
ctrl_d = quick_portfolio_metrics(ret_d, "D) Random-sign (seed=42)")

controls_df = pd.DataFrame([ctrl_a, ctrl_b, ctrl_c, ctrl_d]).set_index("Variant")
controls_df.to_csv(output_dir / "debug_controls.csv")

print("=" * 70)
print("  CONTROL TESTS — SIGNAL vs SCALING DECOMPOSITION")
print("=" * 70)
fmt_ctrl = controls_df.copy()
for c in ["Ann. Return", "Ann. Vol", "Max DD"]:
    fmt_ctrl[c] = fmt_ctrl[c].map("{:.1%}".format)
fmt_ctrl["Sharpe"] = fmt_ctrl["Sharpe"].map("{:.2f}".format)
fmt_ctrl["Calmar"] = fmt_ctrl["Calmar"].map("{:.2f}".format)
display(fmt_ctrl)

if controls_df.loc["A) Baseline TSMOM", "Sharpe"] <= controls_df.loc["D) Random-sign (seed=42)", "Sharpe"]:
    print("\n  *** WARNING: Baseline Sharpe <= Random — signal may have no edge ***")
else:
    delta = controls_df.loc["A) Baseline TSMOM", "Sharpe"] - controls_df.loc["D) Random-sign (seed=42)", "Sharpe"]
    print(f"\n  Signal contributes +{delta:.2f} Sharpe above random.")

print(f"\nExported: debug_controls.csv")

## DEBUG 9 — Robustness / Sensitivity Grid

Test lookback x rebalance frequency combinations.
No parameter tuning — just checking stability.

In [ ]:
lookback_grid = [126, 252, 504]
rebal_grid    = ["M", "Q"]

sensitivity_rows = []
for lb in lookback_grid:
    for rb in rebal_grid:
        r12 = prices_df.pct_change(lb)
        sig = np.sign(r12).shift(1)
        sig_m = sig.resample(rb).last().shift(1)
        sig_d = sig_m.reindex(prices_df.index).ffill().fillna(0.0)
        # Zero pre-listing (Fix 3)
        for col in sig_d.columns:
            fvd = FIRST_VALID_DATE.get(col)
            if fvd is not None:
                min_date = fvd + pd.Timedelta(days=lb + 35)
                sig_d.loc[sig_d.index < min_date, col] = 0.0

        vol_m = vol_ann.resample(rb).last().shift(1)
        vol_d = vol_m.reindex(prices_df.index).ffill().replace(0, np.nan)
        w_raw = sig_d * (TARGET_VOL / vol_d)
        w_cap = w_raw.clip(-LEV_CAP, LEV_CAP).fillna(0.0)
        # Apply hard abs cap
        for col in w_cap.columns:
            bucket = INST_BUCKET.get(col, "commodities")
            cap = ABS_WEIGHT_CAP_BY_INSTRUMENT.get(col,
                  ABS_WEIGHT_CAP_BY_BUCKET.get(bucket, 1.0))
            w_cap[col] = w_cap[col].clip(-cap, cap)

        w_bucket = w_cap * bucket_scaling
        r_pre = (w_bucket.shift(1) * ret).sum(axis=1)
        pv = r_pre.rolling(60, min_periods=20).std() * sqrt(252)
        sc = (PORTFOLIO_VOL_TARGET / pv.replace(0, np.nan)).clip(
             PORTFOLIO_SCALE_BOUNDS[0], PORTFOLIO_SCALE_BOUNDS[1]).fillna(1.0)
        sc_m = sc.resample(rb).last().shift(1)
        sc_d = sc_m.reindex(prices_df.index).ffill().fillna(1.0)
        w_final = w_bucket.multiply(sc_d, axis=0)
        r_port = (w_final.shift(1) * ret).sum(axis=1)
        to = (w_final - w_final.shift(1)).abs().sum(axis=1)
        tc = to * (TC_BP / 10_000)
        r_net = r_port - tc
        m = quick_portfolio_metrics(r_net, f"lb={lb} rb={rb}")
        m["lookback"] = lb
        m["rebalance"] = rb
        sensitivity_rows.append(m)

sensitivity_df = pd.DataFrame(sensitivity_rows)
sensitivity_df.to_csv(output_dir / "debug_sensitivity_grid.csv", index=False)

print("=" * 70)
print("  SENSITIVITY GRID (lookback x rebalance)")
print("=" * 70)
for _, row in sensitivity_df.iterrows():
    print(f"  lb={row['lookback']:>3d}  rb={row['rebalance']}  "
          f"Sharpe={row['Sharpe']:.2f}  AnnRet={row['Ann. Return']:.1%}  "
          f"MaxDD={row['Max DD']:.1%}")

best = sensitivity_df.loc[sensitivity_df["Sharpe"].idxmax()]
worst = sensitivity_df.loc[sensitivity_df["Sharpe"].idxmin()]
spread = best["Sharpe"] - worst["Sharpe"]
print(f"\n  Best:  lb={best['lookback']} rb={best['rebalance']} Sharpe={best['Sharpe']:.2f}")
print(f"  Worst: lb={worst['lookback']} rb={worst['rebalance']} Sharpe={worst['Sharpe']:.2f}")
print(f"  Spread: {spread:.2f}  {'(stable)' if spread < 0.5 else '(fragile)'}")

print(f"\nExported: debug_sensitivity_grid.csv")

## DEBUG 10 — Universe Splits & Cap Variants

Test whether performance depends on a single asset class or is sensitive to weight caps.

In [ ]:
splits = {
    "commodities_only": [i for i in final_weights.columns if INST_BUCKET.get(i) == "commodities"],
    "diversifiers_only": [i for i in final_weights.columns if INST_BUCKET.get(i) != "commodities"],
    "full_universe": list(final_weights.columns),
}
caps = [0.5, 1.0, 2.0]

univ_cap_rows = []
for split_name, insts in splits.items():
    if not insts:
        continue
    for cap_val in caps:
        sig_sub = signals[insts]
        vf_sub = vol_frozen_daily[[c for c in insts if c in vol_frozen_daily.columns]]
        w_raw = sig_sub * (TARGET_VOL / vf_sub.replace(0, np.nan))
        w_cap = w_raw.clip(-cap_val, cap_val).fillna(0.0)
        r_pre = (w_cap.shift(1) * ret[insts]).sum(axis=1)
        pv = r_pre.rolling(60, min_periods=20).std() * sqrt(252)
        sc = (PORTFOLIO_VOL_TARGET / pv.replace(0, np.nan)).clip(
             PORTFOLIO_SCALE_BOUNDS[0], PORTFOLIO_SCALE_BOUNDS[1]).fillna(1.0)
        sc_m = sc.resample("M").last().shift(1)
        sc_d = sc_m.reindex(prices_df.index).ffill().fillna(1.0)
        w_final_sub = w_cap.multiply(sc_d, axis=0)
        r_port = (w_final_sub.shift(1) * ret[insts]).sum(axis=1)
        to = (w_final_sub - w_final_sub.shift(1)).abs().sum(axis=1)
        tc = to * (TC_BP / 10_000)
        r_net = r_port - tc
        m = quick_portfolio_metrics(r_net, f"{split_name} cap={cap_val}")
        m["split"] = split_name
        m["cap"] = cap_val
        m["n_instruments"] = len(insts)
        univ_cap_rows.append(m)

univ_cap_df = pd.DataFrame(univ_cap_rows)
univ_cap_df.to_csv(output_dir / "debug_universe_caps.csv", index=False)

print("=" * 70)
print("  UNIVERSE SPLITS x CAP VARIANTS")
print("=" * 70)
for _, row in univ_cap_df.iterrows():
    print(f"  {row['Variant']:40s}  n={row['n_instruments']:2d}  "
          f"Sharpe={row['Sharpe']:.2f}  AnnRet={row['Ann. Return']:.1%}  "
          f"MaxDD={row['Max DD']:.1%}")

print(f"\nExported: debug_universe_caps.csv")

## FIX 4 — Signal Lookahead Verification

Explicit confirmation of the shift logic and a visual spot-check on GC (Gold).
Also computes naive (no-shift) hit rate vs shifted hit rate as a cross-check.

In [ ]:
print("=" * 70)
print("  FIX 4 — SIGNAL LOOKAHEAD VERIFICATION")
print("=" * 70)

print("\n  Shift logic confirmation:")
print("    1. 12M return = prices.pct_change(252)")
print("    2. Signal = sign(ret_12m).shift(1)  # yesterday's signal")
print("    3. Monthly snapshot: sig.resample('M').last().shift(1)")
print("    4. => signal for month T uses 12M return computed at end of month T-2")
print("    5. Weights applied at start of month T")

# Spot-check GC
gc_col = "gc_fut_front"
if gc_col in prices_df.columns and gc_col in signals.columns:
    r12_gc = prices_df[gc_col].pct_change(252)
    sig_gc = signals[gc_col]
    r_gc = ret[gc_col]

    # Pick 5 month-end dates after warmup
    monthly_idx = sig_gc.resample("M").last().index
    sample_months = monthly_idx[monthly_idx >= "2016-06-01"][:5]

    print(f"\n  GC (Gold) spot-check — 5 sample months:")
    print(f"  {'Rebal Date':12s}  {'12M Ret (t-2)':>14s}  {'Signal':>8s}  {'Next Mo Ret':>12s}")
    for dt in sample_months:
        sig_val = sig_gc.loc[dt] if dt in sig_gc.index else np.nan
        # 12M return used was from end of month T-2
        prev_month = dt - pd.DateOffset(months=2)
        closest = r12_gc.index[r12_gc.index <= prev_month]
        r12_used = r12_gc.loc[closest[-1]] if len(closest) > 0 else np.nan
        # Return earned in following month
        next_month = dt + pd.DateOffset(months=1)
        mask = (r_gc.index > dt) & (r_gc.index <= next_month)
        next_ret = r_gc[mask].sum()
        print(f"  {str(dt)[:10]:12s}  {r12_used:+14.4f}  {sig_val:+8.0f}  {next_ret:+12.4f}")

    # Cross-check: naive vs shifted hit rate
    # Naive signal: no shift (would be lookahead)
    r12_naive = prices_df.pct_change(252)
    sig_naive = np.sign(r12_naive)  # NO shift
    sig_naive_m = sig_naive.resample("M").last()  # NO shift
    sig_naive_d = sig_naive_m.reindex(prices_df.index).ffill().fillna(0.0)

    # Hit rate comparison
    print(f"\n  Hit rate comparison (shifted vs naive/lookahead):")
    for col in signals.columns[:5]:
        r_col = ret[col]
        # Shifted
        hit_shifted = ((signals[col].shift(1) * r_col) > 0).mean()
        # Naive (lookahead)
        hit_naive = ((sig_naive_d[col].shift(1) * r_col) > 0).mean()
        label = LABELS.get(col, col)
        print(f"    {label:22s}  shifted={hit_shifted:.1%}  naive={hit_naive:.1%}  "
              f"delta={hit_naive - hit_shifted:+.1%}")

    print("\n  NOTE: If naive > shifted, it does NOT prove lookahead in the shifted version.")
    print("  It likely means the 12M trend signal has weak predictive power in this sample,")
    print("  which is expected for some instruments in some periods.")
else:
    print("  GC not in universe — skipping spot-check.")

## FIX 5 — UST 2Y (TU) Turnover Diagnostic

Investigate the high turnover for TU. Check for bad prints / roll artefacts.
Winsorization was already applied globally in DEBUG 5.

In [ ]:
tu_col = "ust_2y_fut"
print("=" * 70)
print("  FIX 5 — UST 2Y (TU) DIAGNOSTIC")
print("=" * 70)

if tu_col in prices_df.columns:
    tu_ret_raw = ret_pct[tu_col].dropna()
    tu_ret_clean = ret[tu_col].dropna()
    rs20 = tu_ret_raw.rolling(20, min_periods=5).std().replace(0, np.nan)
    z_scores = (tu_ret_raw / rs20).dropna()

    # Top 10 z-score dates
    top10z = z_scores.abs().nlargest(10)
    print("\n  TU pre-clean top 10 |z-score| dates:")
    for dt, zval in top10z.items():
        r_raw = tu_ret_raw.loc[dt]
        r_cln = tu_ret_clean.loc[dt] if dt in tu_ret_clean.index else np.nan
        flag = " ** BAD PRINT" if abs(zval) > 5 else ""
        print(f"    {str(dt)[:10]}  z={zval:+.1f}  ret_raw={r_raw:+.4f}  "
              f"ret_clean={r_cln:+.4f}{flag}")

    # Post-fix turnover
    tu_to = final_weights[tu_col].diff().abs().sum()
    tu_years = len(final_weights) / 252
    tu_ann_to = tu_to / max(tu_years, 0.01)
    print(f"\n  TU post-fix annualised turnover: {tu_ann_to:.1f}x")
    if tu_ann_to > 12:
        print("  ** Still elevated — check if winsorization threshold needs tightening")
    else:
        print("  TU turnover now within expected range")
else:
    print("  TU not in universe — skipping.")

## FIX 6 — Crude Oil (CL) Vol Contribution Diagnostic

Verify CL weight after hard abs cap and per-instrument realized vol contribution.

In [ ]:
print("=" * 70)
print("  FIX 6 — PER-INSTRUMENT REALISED VOL CONTRIBUTION")
print("=" * 70)

# Realised vol from final instrument returns
print("\n  Per-instrument realised vol (from final_weights * ret):")
for col in final_weights.columns:
    r_inst = final_weights[col].shift(1) * ret[col]
    rv = r_inst.std() * sqrt(252)
    max_w = final_weights[col].abs().max()
    label = LABELS.get(col, col)
    bucket = INST_BUCKET.get(col, "?")
    flag = " ** HIGH" if rv > 0.20 else ""
    print(f"  {label:22s} [{bucket:12s}]  realised_vol={rv:.1%}  max|w|={max_w:.3f}{flag}")

cl_col = "cl_fut_front"
if cl_col in final_weights.columns:
    cl_rv = (final_weights[cl_col].shift(1) * ret[cl_col]).std() * sqrt(252)
    cl_cap = ABS_WEIGHT_CAP_BY_INSTRUMENT.get(cl_col, 1.0)
    print(f"\n  CL (Crude Oil):")
    print(f"    Hard cap applied: {cl_cap}")
    print(f"    Max |weight|: {final_weights[cl_col].abs().max():.3f}")
    print(f"    Realised vol contribution: {cl_rv:.1%}")
    print(f"    Note: portfolio overlay scales entire portfolio down when CL is volatile")

## FIX 7 — Bucket Risk Share Verification (Post-Fix)

Recompute and print realised risk shares after all fixes.
Assert commodities share within 15% band of 70% target.

In [ ]:
print("=" * 70)
print("  FIX 7 — BUCKET RISK SHARE VERIFICATION (post all fixes)")
print("=" * 70)

# Recompute from final weights
final_risk = {}
for bucket in universe:
    insts = [i for i in final_weights.columns if INST_BUCKET.get(i) == bucket]
    if insts:
        risk_b = (final_weights[insts].abs() * vol_frozen_daily[insts]).sum(axis=1)
        final_risk[bucket] = risk_b

total_risk_final = sum(final_risk.values())
total_risk_final = total_risk_final.replace(0, np.nan)

print(f"\n  {'Bucket':15s}  {'Target':>8s}  {'Realised':>10s}  {'Delta':>8s}")
print(f"  {'-'*50}")
for bucket in universe:
    if bucket in final_risk:
        share = (final_risk[bucket] / total_risk_final * 100).mean()
        if bucket == "commodities":
            target = COMMODITY_RISK_BUDGET * 100
        else:
            target = DIVERSIFIER_RISK_BUDGET * 100 / max(len([b for b in universe if b != "commodities"]), 1)
            # Actually just show commodity vs diversifier
        if bucket == "commodities":
            target_pct = COMMODITY_RISK_BUDGET * 100
        else:
            target_pct = np.nan
        delta = share - target_pct if not np.isnan(target_pct) else np.nan
        delta_str = f"{delta:+.1f}%" if not np.isnan(delta) else "N/A"
        target_str = f"{target_pct:.0f}%" if not np.isnan(target_pct) else "N/A"
        print(f"  {bucket:15s}  {target_str:>8s}  {share:9.1f}%  {delta_str:>8s}")

# Summary: commodities vs diversifiers
comm_share = (sum(final_risk[b] for b in final_risk if b == "commodities") / total_risk_final * 100).mean()
div_share = 100 - comm_share

print(f"\n  SUMMARY:")
print(f"    Commodities : {comm_share:.1f}%  (target: {COMMODITY_RISK_BUDGET*100:.0f}%)")
print(f"    Diversifiers: {div_share:.1f}%  (target: {DIVERSIFIER_RISK_BUDGET*100:.0f}%)")

# ASSERTION
if abs(comm_share - 70) < 15:
    print(f"\n  ASSERTION PASSED: |{comm_share:.1f}% - 70%| < 15%")
else:
    print(f"\n  ASSERTION WARNING: |{comm_share:.1f}% - 70%| >= 15%")
    print("  Bucket scaling not converging — check bucket scaler implementation")

## FIX 8 — Worst Drawdown Decomposition

Identify the portfolio drawdown start/end dates and decompose by instrument
and bucket. Required for risk department approval and model validation.

Export: `outputs/debug_drawdown_decomposition.csv`

In [ ]:
print("=" * 70)
print("  FIX 8 — WORST DRAWDOWN DECOMPOSITION")
print("=" * 70)

# Find worst drawdown period
rm = port_equity.cummax()
dd = (port_equity - rm) / rm
trough_date = dd.idxmin()
trough_val = dd.min()

# Find peak (start of drawdown)
peak_date = rm.loc[:trough_date].idxmax()

# Find recovery (if any)
post_trough = port_equity.loc[trough_date:]
recovery_mask = post_trough >= rm.loc[peak_date]
if recovery_mask.any():
    recovery_date = recovery_mask.idxmax()
else:
    recovery_date = port_equity.index[-1]

print(f"  Worst drawdown: {trough_val:.1%}")
print(f"  Peak:     {str(peak_date)[:10]}  (equity: ${port_equity.loc[peak_date]:,.0f})")
print(f"  Trough:   {str(trough_date)[:10]}  (equity: ${port_equity.loc[trough_date]:,.0f})")
print(f"  Recovery: {str(recovery_date)[:10]}")
print(f"  Duration: {(trough_date - peak_date).days} days to trough")

# Per-instrument contribution during drawdown
dd_mask = (ret.index >= peak_date) & (ret.index <= trough_date)
dd_ret = ret[dd_mask]
dd_weights = final_weights[dd_mask]

dd_rows = []
print(f"\n  Per-instrument contribution (peak to trough):")
print(f"  {'Instrument':22s}  {'Bucket':12s}  {'Contrib':>8s}  {'Avg Wt':>8s}  {'Direction':>10s}")
print(f"  {'-'*65}")

for col in final_weights.columns:
    inst_pnl = (dd_weights[col].shift(1) * dd_ret[col]).sum()
    avg_wt = dd_weights[col].mean()
    direction = "LONG" if avg_wt > 0.01 else "SHORT" if avg_wt < -0.01 else "FLAT"
    label = LABELS.get(col, col)
    bucket = INST_BUCKET.get(col, "?")
    print(f"  {label:22s}  {bucket:12s}  {inst_pnl:+8.4f}  {avg_wt:+8.3f}  {direction:>10s}")
    dd_rows.append({
        "instrument": col, "label": label, "bucket": bucket,
        "pnl_contribution": inst_pnl, "avg_weight": avg_wt, "direction": direction,
    })

# Bucket aggregation
print(f"\n  Bucket contributions:")
dd_df = pd.DataFrame(dd_rows)
for bucket in universe:
    sub = dd_df[dd_df["bucket"] == bucket]
    if len(sub) > 0:
        total_pnl = sub["pnl_contribution"].sum()
        print(f"    {bucket:15s}: {total_pnl:+.4f}")

dd_df.to_csv(output_dir / "debug_drawdown_decomposition.csv", index=False)
print(f"\nExported: debug_drawdown_decomposition.csv")

## FIX 9 — Final Turnover Validation

Recompute turnover from FINAL weights after all fixes.
Validate per-instrument and portfolio-level turnover.

In [ ]:
print("=" * 70)
print("  FIX 9 — FINAL TURNOVER VALIDATION")
print("=" * 70)

n_years = len(final_weights) / 252

print("\n  Per-instrument annualised turnover:")
for col in final_weights.columns:
    to = final_weights[col].diff().abs().sum() / max(n_years, 0.01)
    label = LABELS.get(col, col)
    flag = " ** HIGH" if to > 12 else ""
    print(f"    {label:22s}  {to:6.1f}x{flag}")

# Portfolio turnover
port_to_final = turnover_daily.sum() / max(n_years, 0.01)
print(f"\n  Portfolio annualised turnover: {port_to_final:.1f}x")

# Top 20 turnover days
top20 = turnover_daily.nlargest(20)
n_on_boundary = sum(1 for dt in top20.index if month_boundary.loc[dt])
print(f"\n  Top 20 turnover days: {n_on_boundary}/20 on month boundaries")
print(f"  % on month-end: {n_on_boundary/20*100:.0f}%")

for dt, val in top20.items():
    is_mb = "MB" if month_boundary.loc[dt] else "  "
    print(f"    {str(dt)[:10]}  {is_mb}  turnover={val:.6f}")

# FINAL ASSERTION 4
if port_to_final < 100:
    print(f"\n  FINAL ASSERTION 4 PASSED: portfolio turnover {port_to_final:.1f}x < 100x")
else:
    print(f"\n  FINAL ASSERTION 4 FAILED: portfolio turnover {port_to_final:.1f}x >= 100x")
    print("  Daily churn persists — overlay scale may be changing daily")

if port_to_final < 50:
    print(f"  Target check: {port_to_final:.1f}x < 50x target — OK")
elif port_to_final < 30:
    print(f"  Target check: {port_to_final:.1f}x < 30x target — EXCELLENT")

## FIX 10 — Updated Portfolio Metrics + Target Comparison

Final metrics after ALL fixes. Compare against section 6.3 targets.

Export: `outputs/tsmom_metrics_post_fix.csv`, `outputs/tsmom_summary_post_fix.html`

In [ ]:
print("=" * 75)
print("  FIX 10 — FINAL METRICS vs TARGETS (section 6.3)")
print("=" * 75)

pm = port_metrics
realised_vol = port_ret_net.rolling(60, min_periods=20).std() * sqrt(252)

comparison = pd.DataFrame({
    "Metric": [
        "Sharpe Ratio", "Annualised Vol", "Max Drawdown",
        "Calmar Ratio", "Hit Rate", "Ann. Turnover",
    ],
    "Target": [
        f"> {targets['sharpe_per_strategy']:.1f}",
        f"{targets['vol_range_annual'][0]:.0%} - {targets['vol_range_annual'][1]:.0%}",
        f"< {targets['max_drawdown_pct']:.0f}%",
        f"> {targets['calmar_ratio']:.1f}",
        f"> {targets['hit_rate_daily']:.0%}",
        f"< {targets['max_turnover_annual']:.0f}x",
    ],
    "TSMOM (post-fix)": [
        f"{pm['Sharpe']:.2f}",
        f"{pm['Ann. Vol']:.1%}",
        f"{pm['Max DD']:.1%}",
        f"{pm['Calmar']:.2f}",
        f"{pm['Hit Rate']:.1%}",
        f"{pm['Ann. Turnover']:.1f}x",
    ],
    "Status": [
        "PASS" if pm["Sharpe"] > targets["sharpe_per_strategy"] else "MISS",
        "PASS" if targets["vol_range_annual"][0] <= pm["Ann. Vol"] <= targets["vol_range_annual"][1] else "MISS",
        "PASS" if pm["Max DD"] * 100 < targets["max_drawdown_pct"] else "MISS",
        "PASS" if pm["Calmar"] > targets["calmar_ratio"] else "MISS",
        "PASS" if pm["Hit Rate"] > targets["hit_rate_daily"] else "MISS",
        "PASS" if pm["Ann. Turnover"] < targets["max_turnover_annual"] else "MISS",
    ],
}).set_index("Metric")

display(comparison)

# Export metrics
metrics_df.to_csv(output_dir / "tsmom_metrics_post_fix.csv")

# FINAL ASSERTION 2: post-overlay vol
mean_post_vol = realised_vol.dropna().mean()
print(f"\n  FINAL ASSERTION 2: mean post-overlay vol = {mean_post_vol:.1%}")
if 0.06 <= mean_post_vol <= 0.14:
    print("  PASSED")
else:
    print(f"  WARNING: outside [6%, 14%]")

# FINAL ASSERTION 3: SOFR weights zero before 2018-05-04
sofr_check = pd.Timestamp("2018-05-04")
sofr_insts = [c for c in final_weights.columns if c.startswith("sofr_")]
all_zero = True
for inst in sofr_insts:
    n_nz = (final_weights.loc[final_weights.index < sofr_check, inst].abs() > 1e-10).sum()
    if n_nz > 0:
        all_zero = False
        print(f"  FINAL ASSERTION 3 FAIL: {inst} has {n_nz} non-zero weights before 2018-05-04")
if all_zero:
    print(f"  FINAL ASSERTION 3 PASSED: all SOFR weights zero before 2018-05-04")

print(f"\nExported: tsmom_metrics_post_fix.csv")

## Diagnostics

Turnover sanity, portfolio vol realised, correlation structure.

In [ ]:
print("=" * 65)
print("  TURNOVER DIAGNOSTICS")
print("=" * 65)
print(f"  Total turnover (sum of |dw|)     : {turnover_daily.sum():.2f}")
print(f"  Annualised turnover              : {port_metrics['Ann. Turnover']:.1f}x")
print(f"  Days with turnover > 0           : {(turnover_daily > 1e-10).sum()} "
      f"of {len(turnover_daily)} ({(turnover_daily > 1e-10).mean():.1%})")

print(f"\n  Realised portfolio vol (60d rolling):")
print(f"    Mean   : {realised_vol.dropna().mean():.1%}")
print(f"    Median : {realised_vol.dropna().median():.1%}")
print(f"    Range  : [{realised_vol.dropna().min():.1%}, {realised_vol.dropna().max():.1%}]")

corr = ret[active_instruments].rename(columns=LABELS).corr()
print(f"\n  Average pairwise correlation: {corr.values[np.triu_indices_from(corr.values, k=1)].mean():.3f}")

## Visualisations

1. Portfolio equity (net) + rolling vol
2. Per-instrument equity curves
3. Monthly signal heatmap
4. Bucket risk share over time
5. Turnover over time

In [ ]:
# --- 1. Portfolio Equity + Rolling Vol ---
fig_port = make_subplots(
    rows=2, cols=1, shared_xaxes=True, row_heights=[0.65, 0.35],
    subplot_titles=["Portfolio Equity (net of costs)", "Rolling 60d Vol (annualised)"],
    vertical_spacing=0.08,
)
fig_port.add_trace(go.Scatter(
    x=port_equity.index, y=port_equity, name="Portfolio",
    line=dict(color="#2c3e50", width=2),
    fill="tozeroy", fillcolor="rgba(44, 62, 80, 0.08)",
), row=1, col=1)
fig_port.add_trace(go.Scatter(
    x=realised_vol.index, y=realised_vol, name="Realised Vol",
    line=dict(color="#e67e22", width=1.5),
), row=2, col=1)
fig_port.add_hline(y=PORTFOLIO_VOL_TARGET, line_dash="dash", line_color="red",
                   annotation_text="10% target", row=2, col=1)
fig_port.update_layout(
    title="Canonical TSMOM — Portfolio (IS: 2015-2022, post-fix, $1M start)",
    template="plotly_white", height=600, hovermode="x unified", showlegend=False,
)
fig_port.update_yaxes(title_text="Equity ($)", tickformat="$,.0f", row=1, col=1)
fig_port.update_yaxes(title_text="Vol", tickformat=".0%", row=2, col=1)
fig_port.show()

# --- 2. Per-Instrument Equity Curves ---
eq_df = pd.DataFrame({LABELS.get(k,k): v for k, v in inst_equity.items()})
eq_norm = eq_df / eq_df.iloc[0]
bucket_colors = {"commodities": "#f39c12", "rates": "#3498db",
                 "equities": "#2ecc71", "fx": "#9b59b6"}
fig_inst = go.Figure()
for col_orig in final_weights.columns:
    label = LABELS.get(col_orig, col_orig)
    bucket = INST_BUCKET.get(col_orig, "?")
    if label in eq_norm.columns:
        fig_inst.add_trace(go.Scatter(
            x=eq_norm.index, y=eq_norm[label], name=label,
            line=dict(color=bucket_colors.get(bucket, "#95a5a6"), width=1.2),
            legendgroup=bucket, legendgrouptitle_text=bucket,
        ))
fig_inst.update_layout(
    title="Per-Instrument Equity (normalised to $1)",
    template="plotly_white", height=550, hovermode="x unified",
    legend=dict(orientation="h", y=-0.2),
    yaxis_title="Growth of $1", yaxis_tickformat="$.2f",
)
fig_inst.show()

# --- 3. Monthly Signal Heatmap ---
sig_monthly = signals.resample("M").last().rename(columns=LABELS)
fig_heat = px.imshow(
    sig_monthly.T,
    color_continuous_scale=[[0, "#e74c3c"], [0.5, "#ecf0f1"], [1, "#2ecc71"]],
    zmin=-1, zmax=1,
    title="Monthly Signal Regime (+1 Long / -1 Short)",
    labels={"x": "", "y": "Instrument", "color": "Signal"}, aspect="auto",
)
fig_heat.update_layout(template="plotly_white", height=500,
                       xaxis=dict(dtick="M3", tickformat="%Y-%m"))
fig_heat.show()

# --- 4. Bucket Risk Share ---
fig_bucket = go.Figure()
comm_smooth = bucket_risk_shares["commodities_pct"].rolling(21).mean()
div_smooth = bucket_risk_shares["diversifiers_pct"].rolling(21).mean()
fig_bucket.add_trace(go.Scatter(x=comm_smooth.index, y=comm_smooth,
    name="Commodities %", line=dict(color="#f39c12", width=2)))
fig_bucket.add_trace(go.Scatter(x=div_smooth.index, y=div_smooth,
    name="Diversifiers %", line=dict(color="#3498db", width=2)))
fig_bucket.add_hline(y=COMMODITY_RISK_BUDGET*100, line_dash="dash",
                     line_color="#f39c12", annotation_text="70% target")
fig_bucket.add_hline(y=DIVERSIFIER_RISK_BUDGET*100, line_dash="dash",
                     line_color="#3498db", annotation_text="30% target")
fig_bucket.update_layout(
    title="Bucket Risk Share Over Time (21d smoothed)",
    template="plotly_white", height=400, hovermode="x unified",
    yaxis_title="Risk Share (%)", yaxis_range=[0, 100],
)
fig_bucket.show()

# --- 5. Turnover ---
fig_turn = go.Figure()
fig_turn.add_trace(go.Bar(
    x=turnover_daily.index, y=turnover_daily,
    marker_color="#3498db", opacity=0.7, name="Turnover",
))
fig_turn.update_layout(
    title="Daily Turnover (should spike at month boundaries only)",
    template="plotly_white", height=350,
    yaxis_title="Turnover (sum |dw|)", hovermode="x unified",
)
fig_turn.show()

## Export

Save signals, weights, equity, bucket risk, and summary (post-fix versions).

In [ ]:
output_dir = Path("../outputs")
output_dir.mkdir(exist_ok=True)

datestamp = datetime.now().strftime("%Y%m%d")

sig_path = output_dir / f"tsmom_signals_monthly_{datestamp}.csv"
signals.rename(columns=LABELS).to_csv(sig_path)

w_path = output_dir / f"tsmom_weights_daily_{datestamp}.csv"
weights_scaled.rename(columns=LABELS).to_csv(w_path)

wf_path = output_dir / f"tsmom_weights_final_daily_{datestamp}.csv"
final_weights.rename(columns=LABELS).to_csv(wf_path)

eq_path = output_dir / f"tsmom_portfolio_equity_{datestamp}.csv"
eq_export = pd.DataFrame({
    "portfolio_equity": port_equity, "portfolio_ret_net": port_ret_net,
    "portfolio_ret_gross": port_ret_gross, "turnover": turnover_daily,
})
eq_export.to_csv(eq_path)

br_path = output_dir / f"tsmom_bucket_risk_shares_{datestamp}.csv"
bucket_risk_shares.to_csv(br_path)

# Post-fix metrics
mf_path = output_dir / f"tsmom_metrics_post_fix_{datestamp}.csv"
metrics_df.to_csv(mf_path)

# Post-fix equity
eqf_path = output_dir / f"tsmom_portfolio_equity_post_fix_{datestamp}.csv"
eq_export.to_csv(eqf_path)

# Drawdown decomposition (already exported in Fix 8 cell)

# Summary HTML
html_path = output_dir / f"tsmom_summary_post_fix_{datestamp}.html"
html_content = (
    "<h2>Canonical TSMOM — IS Performance (2015-2022, POST-FIX)</h2>\n"
    "<p>Signal: sign(12M return) | Monthly rebalance | EWMA vol (lambda=0.94) | "
    f"Commodity budget: {COMMODITY_RISK_BUDGET:.0%} | Portfolio vol target: {PORTFOLIO_VOL_TARGET:.0%}<br>"
    f"Overlay scale floor: {PORTFOLIO_SCALE_BOUNDS[0]} | Abs weight caps: bucket-specific | "
    f"Winsorisation: {WINSOR_Z_THRESHOLD}z</p>\n"
    "<h3>Per-instrument + Portfolio Metrics</h3>\n"
    + fmt_df.to_html()
    + "<br><h3>vs Memory File Targets (section 6.3)</h3>\n"
    + comparison.to_html()
    + "<br><h3>Control Tests</h3>\n"
    + controls_df.to_html()
)
with open(html_path, "w") as f:
    f.write(html_content)

print(f"Exported to {output_dir.resolve()}/")
for p in [sig_path, w_path, wf_path, eq_path, br_path, mf_path, eqf_path, html_path]:
    print(f"  {p.name}")

# Debug exports reminder
print(f"\nDebug exports (from earlier cells):")
debug_files = ["debug_coverage.csv", "debug_units_audit.csv", "debug_return_stats.csv",
               "debug_top_shocks.csv", "debug_roll_flags.csv", "debug_vol_summary.csv",
               "debug_weights_summary.csv", "debug_turnover.csv", "debug_controls.csv",
               "debug_sensitivity_grid.csv", "debug_universe_caps.csv",
               "debug_drawdown_decomposition.csv"]
for f in debug_files:
    print(f"  {f}")

print(f"\n" + "=" * 65)
print(f"  ALL FIXES APPLIED. Notebook complete: {datetime.now():%Y-%m-%d %H:%M}")
print(f"=" * 65)